In [1]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"


In [2]:
import cv2
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix
from skimage import feature 
import os 
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV 


In [3]:
# Function to extract LBP features from an image
def compute_lbp(image):
    lbp = feature.local_binary_pattern(image, P=16, R=2, method="uniform")
    hist, _ = np.histogram(lbp.ravel(), bins=np.arange(0, 58), range=[0, 58])
    hist = hist.astype("float")
    hist /= (hist.sum() + 1e-7)
    return hist 


In [4]:
# Function to read images and extract features
def extract_features_and_labels(data_folder):
    data = []
    labels = []

    for label, folder in enumerate(["Fake", "Live"]):
        folder_path = f"dataset/{data_folder}/{folder}"
        for filename in os.listdir(folder_path):
            if filename.endswith(".png"):
                image_path = os.path.join(folder_path, filename)
                image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
                lbp_features = compute_lbp(image)
                data.append(lbp_features)
                labels.append(label)

    return np.array(data), np.array(labels) 


# train 


In [5]:
# Read training data and extract features
data_folder = "training"
X_train, y_train = extract_features_and_labels(data_folder) 

# Apply feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# Use grid search to find the best hyperparameters
param_grid = {'C': [0.1, 1, 10, 100], 'gamma': [0.1, 0.01, 0.001, 0.0001]}
grid_search = GridSearchCV(SVC(kernel='rbf'), param_grid, cv=5)
grid_search.fit(X_train_scaled, y_train)

# Get the best SVM classifier
best_svm_classifier = grid_search.best_estimator_

# Predict on training data
train_predictions = best_svm_classifier.predict(X_train_scaled)

# Evaluate on training data
train_accuracy = accuracy_score(y_train, train_predictions)
train_conf_matrix = confusion_matrix(y_train, train_predictions)

print("Training Accuracy:", train_accuracy)
print("Confusion Matrix (Training):")
print(train_conf_matrix)


GridSearchCV(cv=5, estimator=SVC(),
             param_grid={'C': [0.1, 1, 10, 100],
                         'gamma': [0.1, 0.01, 0.001, 0.0001]})

Training Accuracy: 0.9901719901719902
Confusion Matrix (Training):
[[206   1]
 [  3 197]]


# test 


In [6]:
# Read testing data and extract features
data_folder = "testing"
X_test, y_test = extract_features_and_labels(data_folder) 

# Applying the same feature scaler 
X_test_scaled = scaler.fit_transform(X_test) 

# Predict on testing data
test_predictions = best_svm_classifier.predict(X_test_scaled)

# Evaluate on testing data
test_accuracy = accuracy_score(y_test, test_predictions)
test_conf_matrix = confusion_matrix(y_test, test_predictions)

print("Testing Accuracy:", test_accuracy)
print("Confusion Matrix (Testing):")
print(test_conf_matrix)


Testing Accuracy: 0.83
Confusion Matrix (Testing):
[[159  41]
 [ 27 173]]
